# Parte 1 - Baseline RAG funcional

## Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña

**Implementa el pipeline sobre tu corpus, a partir del Lab-02 del día miércoles:**

In [2]:
import socket
import sys
from pathlib import Path
import sys
import os
from qdrant_client import QdrantClient
import pandas as pd
import json


In [3]:

# Buscar la raíz del proyecto
raiz = Path.cwd()

while raiz != raiz.parent:
    if (
        (raiz / "pyproject.toml").exists()
        and (raiz / "rag_pipeline.py").exists()
    ):
        break
    raiz = raiz.parent

ROOT = raiz

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)
print("rag_pipeline.py:", (ROOT / "rag_pipeline.py").exists())
print("pyproject.toml:", (ROOT / "pyproject.toml").exists())

Raíz del proyecto: c:\Users\ASUS\Desktop\Maestria IA USFQ\IA Generativa y Agentes\Semana2\Taller2\taller-02-rag-alvarez-ballesteros-ludena
rag_pipeline.py: True
pyproject.toml: True


In [4]:
from rag_pipeline import RagPipeline

In [5]:
## definimos host y direcciones

os.environ["EMBEDDING_BACKEND"] = "h200"

os.environ["QDRANT_URL"] = "http://localhost:6333"

# Colección específica del baseline
os.environ["QDRANT_COLLECTION"] = "mmia6013_rag_baseline"

# Corpus real
os.environ["CORPUS_DIR"] = str(ROOT / "corpus")

os.environ["OLLAMA_URL"] = "http://localhost:11434"

print("EMBEDDING_BACKEND:", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL:", os.environ["QDRANT_URL"])
print("CORPUS_DIR:", os.environ["CORPUS_DIR"])

EMBEDDING_BACKEND: h200
QDRANT_URL: http://localhost:6333
CORPUS_DIR: c:\Users\ASUS\Desktop\Maestria IA USFQ\IA Generativa y Agentes\Semana2\Taller2\taller-02-rag-alvarez-ballesteros-ludena\corpus


In [6]:
cliente_qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"]
)

print("Colecciones existentes:")
print(cliente_qdrant.get_collections())

Colecciones existentes:


c:\Users\ASUS\Desktop\Maestria IA USFQ\IA Generativa y Agentes\Semana2\Taller2\taller-02-rag-alvarez-ballesteros-ludena\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:292: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


ResponseHandlingException: timed out

In [ ]:
print("Host:", socket.gethostname())


In [ ]:
pipeline = RagPipeline(
    collection="mmia6013_rag_baseline"
)

print("Backend:", os.environ["EMBEDDING_BACKEND"])
print("Modelo:", pipeline.encoder.modelo if hasattr(pipeline.encoder, "modelo") else "bge-m3")
print("Tope del modelo:", pipeline.max_seq_tokens)
print("Colección:", pipeline.collection)

#### 1. Ingesta. Parsea los documentos (PDF → texto). Documenta qué se pierde o se degrada (tablas, encabezados,pies de página) y qué documentos, si alguno, la ingesta rechazó por no tener texto útil.

In [ ]:
corpus_dir = ROOT / "corpus"

print("Corpus:", corpus_dir)
print("¿Existe?:", corpus_dir.exists())
print("¿Es carpeta?:", corpus_dir.is_dir())

documentos = sorted(corpus_dir.glob("*"))

print("\nDocumentos encontrados:")
for doc in documentos:
    print(" -", doc.name)

In [ ]:
chunks = pipeline.ingest(
    corpus_dir,
    chunk_tokens=512,
    overlap_tokens=102
)

print(f"\nFragmentos obtenidos: {len(chunks)}")

In [ ]:


datos_chunks = []

for chunk in chunks:
    datos_chunks.append({
        "chunk_id": chunk.id,
        "documento": chunk.document,
        "chunk_index": chunk.chunk_index,
        "tokens": len(
            pipeline.encoder.tokenizer(
                chunk.text,
                add_special_tokens=False,
                truncation=False
            )["input_ids"]
        ),
        "texto": chunk.text
    })

df_chunks = pd.DataFrame(datos_chunks)
print("Total chunks:", len(df_chunks))

In [ ]:
df_chunks.head()

In [ ]:
carpeta_resultados = ROOT / "resultados" / "parte1"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo_chunks = carpeta_resultados / "numeral1.csv"

df_chunks.to_csv(
    archivo_chunks,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_chunks}")

### 2. Fragmentación. Baseline de tamaño fijo con solapamiento, en tokens del modelo de embeddings, con `chunk_tokens` y  `overlap_tokens` explícitos. El tope de bge-m3 es 8192 y el  Lab-02 del día miércoles corta por defecto en 512: la justificación dice por qué tu tamaño sirve a tus preguntas, no solo que cabe.

In [ ]:
CHUNK_TOKENS = 512
OVERLAP_TOKENS = 102

print("Parámetros de fragmentación")
print("chunk_tokens:", CHUNK_TOKENS)
print("overlap_tokens:", OVERLAP_TOKENS)
print("máximo bge-m3:", pipeline.max_seq_tokens)

Se utiliza un tamaño de 512 tokens con 102 tokens de solapamiento. Este tamaño sigue el baseline propuesto por el Lab-02 y mantiene los fragmentos suficientemente por debajo del límite de 8192 tokens de bge-m3. El solapamiento permite conservar parte del contexto cuando una idea se encuentra cerca del límite entre dos fragmentos. La elección será evaluada posteriormente mediante el golden set, por lo que no se asume de antemano que 512 sea óptimo.

Se utiliza un tamaño de 512 tokens con 102 tokens de solapamiento,
siguiendo el tamaño baseline establecido en el Lab-02. Para este corpus,
un fragmento de 512 tokens permite recuperar una unidad de contenido
suficientemente amplia para responder preguntas conceptuales sobre los
papers, manteniendo una granularidad adecuada para distinguir diferentes
temas dentro de un mismo documento. El solapamiento de 102 tokens
(aproximadamente 20 %) permite conservar contexto cuando una idea o
explicación queda dividida entre dos fragmentos. El tamaño se mantiene
muy por debajo del límite de 8192 tokens de bge-m3, pero esta condición
solo garantiza que el fragmento pueda ser procesado; la elección final
será evaluada posteriormente con el golden set mediante Hit Rate y MRR.

###  3. Embedding e indexación. Con bge-m3 en la H200 (EMBEDDING_BACKEND=h200, el valor por defecto, con GlobalProtect conectada) o, sin VPN, con OpenAI (EMBEDDING_BACKEND=openai). Declara en el informe la fila por su id y con su verified_at. Indexa en Qdrant (el contenedor, no :memory:, que es solo para depurar).

In [ ]:
pipeline.index(chunks)

print("Índice creado correctamente.")
print("Colección:", pipeline.collection)
print("Número de chunks indexados:", len(chunks))

In [ ]:
info = cliente_qdrant.get_collection(
    collection_name=pipeline.collection
)

print(info)

In [ ]:
cantidad = cliente_qdrant.count(
    collection_name=pipeline.collection,
    exact=True
)

print("Vectores en Qdrant:", cantidad.count)

In [ ]:
df_embedding = pd.DataFrame([{
    "id": "embed_local_multilingue",
    "proveedor": "BAAI",
    "modelo": "BAAI/bge-m3",
    "dimension": 1024,
    "tokens_secuencia": 8192,
    "multilingue": True,
    "costo_usd_1M": 0,
    "verified_at": "2026-08-27"
}])

display(df_embedding)

In [ ]:
archivo_embedding = carpeta_resultados / "numeral3_embedding_baseline.csv"

df_embedding.to_csv(
    archivo_embedding,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_embedding}")

### 4. Recuperación. Muestra los top-5 fragmentos para al menos 3 preguntas de prueba, con documento y puntaje.

In [ ]:
preguntas_prueba = [
    "What are the main advantages of the Transformer architecture compared with recurrent and convolutional neural networks for sequence transduction?",

    "How does GPT-3 demonstrate that increasing model size and using in-context learning can improve few-shot performance?",

    "How does InstructGPT use human feedback to improve the alignment of language models with user intentions?"
]

In [ ]:
resultados_retrieval = []

for pregunta in preguntas_prueba:

    resultados = pipeline.retrieve(
        pregunta,
        top_k=5
    )

    print("=" * 100)
    print("PREGUNTA:")
    print(pregunta)
    print("\nTOP 5 VECINOS:\n")

    for rank, resultado in enumerate(resultados, start=1):

        print(f"--- Vecino {rank} ---")
        print(f"Score:     {resultado['score']:.4f}")
        print(f"Documento: {resultado['document']}")
        print(f"Chunk ID:  {resultado['chunk_id']}")
        print(f"Texto:    {resultado['text']}")
        print()

        resultados_retrieval.append({
            "pregunta": pregunta,
            "rank": rank,
            "score": resultado["score"],
            "documento": resultado["document"],
            "chunk_id": resultado["chunk_id"],
            "texto": resultado["text"]
        })

In [ ]:
df_retrieval = pd.DataFrame(resultados_retrieval)

display(df_retrieval)

In [ ]:
archivo_retrieval = carpeta_resultados / "numeral4_retrieval_top5.csv"

df_retrieval.to_csv(
    archivo_retrieval,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_retrieval}")

### 5. Generación.`rag_pipeline.generate` ya trae la llamada: con la clave del curso usa la fila propietario_economico; sin clave, Ollama con open_weight_pequeno. El prompt del baseline hace tres cosas—delimita el contexto, obliga a usar solo el contexto y autoriza la abstención con una frase fija— y esa frase es la misma en el prompt, en el golden set y en el detector: ABSTENCION en cambiarla en los tres sitios

In [ ]:
resultados_generacion = []

for pregunta in preguntas_prueba:

    resultado = pipeline.answer(
        pregunta,
        top_k=5
    )

    print("=" * 100)
    print("PREGUNTA:")
    print(pregunta)

    print("\nGENERADOR:")
    print(resultado["generator"])

    print("\nABSTENCIÓN:")
    print(resultado["abstained"])

    print("\nRESPUESTA:")
    print(resultado["answer"])

    print("\nPROMPT UTILIZADO:")
    print(resultado["prompt"])

    print()

    resultados_generacion.append({
        "pregunta": resultado["question"],
        "respuesta": resultado["answer"],
        "abstencion": resultado["abstained"],
        "generador": resultado["generator"],
        "prompt": resultado["prompt"]
    })

In [ ]:
df_generacion = pd.DataFrame(resultados_generacion)

display(df_generacion[
    [
        "pregunta",
        "respuesta",
        "abstencion",
        "generador"
    ]
])

In [ ]:
archivo_generacion = carpeta_resultados / "numeral5_generacion.csv"

df_generacion.to_csv(
    archivo_generacion,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_generacion}")

In [ ]:
salidas_generacion = []

for pregunta in preguntas_prueba:

    resultado = pipeline.answer(
        pregunta,
        top_k=5
    )

    salidas_generacion.append(resultado)

archivo_generacion_json = carpeta_resultados / "numeral5_generacion.json"

with open(
    archivo_generacion_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        salidas_generacion,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Guardado en: {archivo_generacion_json}")